In [1]:
# 1. IMPORTS
import requests
import pandas as pd
import time # Sử dụng để tạo độ trễ (sleep) tránh bị chặn API

# 2. CONFIGURATIONS
LATITUDE = 21.0050
LONGITUDE = 106.8333

# Cấu hình danh sách các năm để gọi API Thời tiết làm 3 lần
WEATHER_PERIODS = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
    ("2026-01-01", "2026-04-10")
]

# Cấu hình khoảng thời gian tổng cho API Khí thải (gọi 1 lần)
AQ_START_DATE = "2022-01-01"
AQ_END_DATE = "2026-03-31"

# Endpoints
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

# Tên file xuất ra cuối cùng
FINAL_OUTPUT_FILE = "data_dongmai_all_raw.csv"

# 3. CORE FUNCTIONS

def fetch_weather_data(lat, lon, start, end):
    """
    Hàm gọi API để lấy dữ liệu Khí tượng (ERA5) theo từng khoảng thời gian.
    Input: Tọa độ (lat, lon) và mốc thời gian (start, end).
    Output: pandas DataFrame chứa dữ liệu thời tiết.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation",
        "timezone": "Asia/Bangkok"
    }
    
    try:
        print(f"  -> Đang tải dữ liệu Khí tượng từ {start} đến {end}...")
        response = requests.get(WEATHER_API_URL, params=params)
        response.raise_for_status() 
        data = response.json()
        
        # Trích xuất và chuyển đổi sang DataFrame
        df = pd.DataFrame(data["hourly"])
        return df
    except requests.exceptions.RequestException as e:
        print(f"Lỗi kết nối API Khí tượng giai đoạn {start}-{end}: {e}")
        return None

def fetch_air_quality_data(lat, lon, start, end):
    """
    Hàm gọi API để lấy dữ liệu Bụi mịn và Khí thải (CAMS) cho toàn bộ 3 năm.
    Input: Tọa độ (lat, lon) và mốc thời gian tổng (start, end).
    Output: pandas DataFrame chứa dữ liệu chất lượng không khí.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": "pm2_5,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth",
        "timezone": "Asia/Bangkok"
    }
    
    try:
        print(f"Đang tải toàn bộ dữ liệu Khí thải & Bụi mịn từ {start} đến {end}...")
        response = requests.get(AIR_QUALITY_API_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        df = pd.DataFrame(data["hourly"])
        return df
    except requests.exceptions.RequestException as e:
        print(f"Lỗi kết nối API Khí thải: {e}")
        return None

# 4. MAIN EXECUTION

def main_data_collection():
    print(f"BẮT ĐẦU THU THẬP DỮ LIỆU KCN ĐÔNG MAI (Tọa độ: {LATITUDE}, {LONGITUDE})")
    
    # Bước 1: Thu thập dữ liệu thời tiết làm 3 lần
    print("\n--- BƯỚC 1: Tải dữ liệu Khí tượng ---")
    weather_dfs = []
    
    for start_date, end_date in WEATHER_PERIODS:
        df_temp = fetch_weather_data(LATITUDE, LONGITUDE, start_date, end_date)
        if df_temp is not None:
            weather_dfs.append(df_temp)
        # Tạm nghỉ 2 giây giữa các lần gọi để tránh bị chặn API do request liên tục
        time.sleep(2)
        
    # Gộp 3 DataFrame thời tiết lại thành 1 bảng tổng
    if weather_dfs:
        df_weather_total = pd.concat(weather_dfs, ignore_index=True)
        print(f"Hoàn thành tải Khí tượng! Tổng số dòng: {len(df_weather_total)}")
    else:
        print("Lỗi: Không tải được dữ liệu Khí tượng.")
        return

    # Bước 2: Thu thập dữ liệu khí thải 1 lần
    print("\n--- BƯỚC 2: Tải dữ liệu Khí thải (Chạy 1 lần) ---")
    df_aq_total = fetch_air_quality_data(LATITUDE, LONGITUDE, AQ_START_DATE, AQ_END_DATE)
    
    if df_aq_total is not None:
        print(f"Hoàn thành tải Khí thải! Tổng số dòng: {len(df_aq_total)}")
    else:
        print("Lỗi: Không tải được dữ liệu Khí thải.")
        return

    # Bước 3: Hợp nhất (Merge) 2 bộ dữ liệu và lưu file
    print("\n--- BƯỚC 3: Hợp nhất dữ liệu và Lưu file ---")
    try:
        # Hợp nhất dựa trên cột 'time' để đảm bảo các thông số khớp chính xác từng giờ
        df_final = pd.merge(df_weather_total, df_aq_total, on="time", how="inner")
        
        # Lưu ra file CSV
        df_final.to_csv(FINAL_OUTPUT_FILE, index=False)
        print(f"THÀNH CÔNG! Đã lưu bản ghi hoàn chỉnh gồm {len(df_final)} dòng vào '{FINAL_OUTPUT_FILE}'.")
        
        # In ra màn hình để kiểm tra trực quan
        print("\nBản xem trước dữ liệu tổng hợp (Head):")
        print(df_final.head())
    except Exception as e:
        print(f"Lỗi trong quá trình hợp nhất hoặc lưu file: {e}")

if __name__ == "__main__":
    main_data_collection()

BẮT ĐẦU THU THẬP DỮ LIỆU KCN ĐÔNG MAI (Tọa độ: 21.005, 106.8333)

--- BƯỚC 1: Tải dữ liệu Khí tượng ---
  -> Đang tải dữ liệu Khí tượng từ 2023-01-01 đến 2023-12-31...
  -> Đang tải dữ liệu Khí tượng từ 2024-01-01 đến 2024-12-31...
  -> Đang tải dữ liệu Khí tượng từ 2025-01-01 đến 2025-12-31...
  -> Đang tải dữ liệu Khí tượng từ 2026-01-01 đến 2026-04-10...
Hoàn thành tải Khí tượng! Tổng số dòng: 28704

--- BƯỚC 2: Tải dữ liệu Khí thải (Chạy 1 lần) ---
Đang tải toàn bộ dữ liệu Khí thải & Bụi mịn từ 2022-01-01 đến 2026-03-31...
Hoàn thành tải Khí thải! Tổng số dòng: 37224

--- BƯỚC 3: Hợp nhất dữ liệu và Lưu file ---
THÀNH CÔNG! Đã lưu bản ghi hoàn chỉnh gồm 28464 dòng vào 'data_dongmai_all_raw.csv'.

Bản xem trước dữ liệu tổng hợp (Head):
               time  temperature_2m  relative_humidity_2m  wind_speed_10m  \
0  2023-01-01T00:00            11.6                    80             8.8   
1  2023-01-01T01:00            11.4                    79             8.1   
2  2023-01-01T02:00 